# ExoSignal Observatory - NASA DR24 Deep Learning Run

End-to-end Colab notebook for the new NASA Kepler DR24 dataset. Run from top to bottom on a T4 GPU. It cleans Colab temporary storage, downloads official NASA DR24 TCE metadata, materializes real Kepler light curves, checkpoints processed parquets, trains the hybrid InceptionTime deep model, evaluates it, and exports artifacts.


In [ ]:
# 1) Runtime + GPU check
import os, sys, json, platform, subprocess, textwrap, time
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working directory before clone:', os.getcwd())

try:
    import tensorflow as tf
    print('TensorFlow:', tf.__version__)
    print('GPU devices:', tf.config.list_physical_devices('GPU'))
except Exception as exc:
    print('TensorFlow import check failed before install:', repr(exc))

!nvidia-smi || true
!df -h /content


In [ ]:
# 2) Hard cleanup of Colab temporary storage
# This does NOT delete Google Drive. It only deletes /content project/cache files from this runtime.
!echo '=== BEFORE CLEANUP: /content ==='
!du -h -d 1 /content 2>/dev/null | sort -h || true
!echo '=== BEFORE CLEANUP: pip/hf caches ==='
!du -sh /root/.cache/pip /root/.cache/huggingface 2>/dev/null || true

!rm -rf   /content/exosignal-observatory   /content/exosignal_*   /content/dr24_*   /content/kepler*   /content/astronet*   /content/lightcurves   /content/tce*   /content/.cache   /root/.cache/pip   /root/.cache/huggingface

!echo '=== AFTER CLEANUP: /content ==='
!du -h -d 1 /content 2>/dev/null | sort -h || true
!df -h /content
print('Note: Colab still shows a large base image/system allocation. That is not project data and cannot be deleted from /content.')


In [ ]:
# 3) Clone latest GitHub repo
REPO_URL = 'https://github.com/Maaskk/exosignal-observatory.git'
PROJECT_DIR = Path('/content/exosignal-observatory')

!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -5
!git status --short


In [ ]:
# 4) Install dependencies
!python -m pip install -q -r requirements-deep.txt

import numpy as np
import pandas as pd
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('No GPU detected. Use Runtime -> Change runtime type -> T4 GPU, then rerun.')


In [ ]:
# 5) Run configuration
# Full NASA predicted-label build gives the largest DR24 source: 20,367 TCE rows.
LABEL_SOURCE = 'predicted'      # 'predicted' = 20,367 weak-label rows; 'clean' = 15,737 human-reviewed rows
SEQUENCE_LENGTH = 1024
TARGET_TIMEOUT_SECONDS = 180    # skip very slow/broken targets instead of blocking the run
CHECKPOINT_ROWS = 50            # write processed train/val/test parquets every 50 materialized TCE rows
MAX_LIGHTCURVE_ROWS = None      # None = full selected label source; set e.g. 1000 only for a quick debug run

TRAIN_EPOCHS = 40
BATCH_SIZE = 32
ENSEMBLE = 3
TTA_RUNS = 8

config = {
    'label_source': LABEL_SOURCE,
    'sequence_length': SEQUENCE_LENGTH,
    'target_timeout_seconds': TARGET_TIMEOUT_SECONDS,
    'checkpoint_rows': CHECKPOINT_ROWS,
    'max_lightcurve_rows': MAX_LIGHTCURVE_ROWS,
    'train_epochs': TRAIN_EPOCHS,
    'batch_size': BATCH_SIZE,
    'ensemble': ENSEMBLE,
    'tta_runs': TTA_RUNS,
}
print(json.dumps(config, indent=2))


In [ ]:
# 6) NASA DR24 metadata-only pass: class distribution and source description
metadata_cmd = [
    'python', 'scripts/build_nasa_dr24_dataset.py',
    '--label-source', LABEL_SOURCE,
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--target-timeout-seconds', str(TARGET_TIMEOUT_SECONDS),
    '--checkpoint-rows', str(CHECKPOINT_ROWS),
]
print('Running:', ' '.join(metadata_cmd))
subprocess.run(metadata_cmd, check=True)

manifest = json.loads(Path('data/nasa_dr24_tce/manifest.json').read_text())
print(json.dumps({
    'metadata_rows': manifest['metadata_rows'],
    'labeled_rows': manifest['labeled_rows'],
    'unique_kepid': manifest['unique_kepid'],
    'class_distribution': manifest['class_distribution'],
    'human_reviewed_rows': manifest['human_reviewed_rows'],
    'source_table': manifest['source']['table'],
}, indent=2))

pd.read_csv('data/nasa_dr24_tce/dr24_tce_labeled.csv').head(10)


In [ ]:
# 7) Build real Kepler light-curve tensors from NASA/MAST via Lightkurve
# This is the long step. Output is visible target by target, and parquets checkpoint every CHECKPOINT_ROWS rows.
build_cmd = [
    'python', '-u', 'scripts/build_nasa_dr24_dataset.py',
    '--label-source', LABEL_SOURCE,
    '--download-lightcurves',
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--target-timeout-seconds', str(TARGET_TIMEOUT_SECONDS),
    '--checkpoint-rows', str(CHECKPOINT_ROWS),
]
if MAX_LIGHTCURVE_ROWS is not None:
    build_cmd += ['--max-lightcurve-rows', str(MAX_LIGHTCURVE_ROWS)]

print('Running:', ' '.join(build_cmd))
start = time.time()
subprocess.run(build_cmd, check=True)
print('Light-curve build elapsed minutes:', round((time.time() - start) / 60, 2))


In [ ]:
# 8) Verify processed NASA DR24 dataset after materialization
processed = Path('data/nasa_dr24_tce/processed')
for split in ['train', 'val', 'test']:
    path = processed / f'{split}.parquet'
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Build step did not finish enough rows/checkpoint.')
    df = pd.read_parquet(path)
    print('
', split, path, 'rows=', len(df), 'size_mb=', round(path.stat().st_size / 1024 / 1024, 2))
    print(df['disposition'].value_counts())

manifest = json.loads(Path('data/nasa_dr24_tce/manifest.json').read_text())
print('
Manifest materialization:')
print(json.dumps(manifest.get('materialization', {}), indent=2)[:4000])
!df -h /content


In [ ]:
# 9) Train on the new NASA DR24 processed dataset
train_cmd = [
    'python', '-u', 'scripts/train_deep_learning.py',
    '--dataset-profile', 'nasa-dr24',
    '--task', 'binary',
    '--model-family', 'hybrid',
    '--backbone', 'inceptiontime',
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--mixed-precision',
    '--augment',
    '--loss', 'focal',
    '--lr-schedule', 'cosine',
    '--ensemble', str(ENSEMBLE),
    '--calibration', 'isotonic',
    '--tta-runs', str(TTA_RUNS),
]
print('Running:', ' '.join(train_cmd))
start = time.time()
subprocess.run(train_cmd, check=True)
print('Training elapsed minutes:', round((time.time() - start) / 60, 2))


In [ ]:
# 10) Final metrics table
metrics = json.loads(Path('models/deep_model_metrics.json').read_text())
summary_keys = [
    'dataset_profile', 'model_family', 'backbone', 'ensemble_members', 'sequence_length',
    'training_rows', 'validation_rows', 'test_rows', 'threshold',
    'roc_auc', 'pr_auc', 'precision', 'recall', 'f1', 'elapsed_seconds'
]
summary = {key: metrics.get(key) for key in summary_keys if key in metrics}
print(json.dumps(summary, indent=2))
print('
Confusion matrix:')
print(json.dumps(metrics.get('confusion_matrix'), indent=2))

pd.DataFrame([summary])


In [ ]:
# 11) Export trained artifacts and reports
import zipfile
artifact_zip = Path('/content/exosignal_nasa_dr24_final_artifacts.zip')
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    patterns = [
        'models/*.json', 'models/*.keras',
        'models/experiments/**/*.keras',
        'reports/*.json', 'reports/*.md',
        'data/nasa_dr24_tce/manifest.json',
        'data/nasa_dr24_tce/dr24_tce_labeled.csv',
    ]
    for pattern in patterns:
        for file in Path('.').glob(pattern):
            zf.write(file, file.as_posix())
print('Artifact zip:', artifact_zip, round(artifact_zip.stat().st_size / 1024 / 1024, 2), 'MB')

try:
    from google.colab import files
    files.download(str(artifact_zip))
except Exception as exc:
    print('Download helper unavailable:', repr(exc))


## Scientific note

This notebook does not claim to confirm exoplanets. The model ranks transit-like signals as exoplanet candidates for follow-up. Final confirmation still requires astrophysical vetting, false-positive checks, and independent observations.
